# setup_kernels

노트북 4개가 쓸 **Jupyter 커널을 서로 격리해서** 만든다.

| 커널 | 노트북 |
|---|---|
| `ace`  | `bgm_ace_step.ipynb` |
| `sa3`  | `bgm_stable_audio.ipynb` |
| `flux` | `image_flux2_klein.ipynb` |
| `img`  | `image_hidream.ipynb` |

## 왜 환경을 4개로 나누는가 — 취향이 아니라 강제다

각 저장소의 `pyproject.toml` 원문을 확인한 결과 요구사항이 서로 배타적이다.

| | ace | sa3 | flux | img |
|---|---|---|---|---|
| python | `>=3.11,<3.13` | `>=3.10` | `>=3.10,<3.13` | 3.12 |
| torch | `2.10.0+cu128` | `2.7.1` (cu126) | `2.8.0` | 2.8.0 |
| **transformers** | `>=4.51,<4.58` | **`>=5.8.0`** | **`==4.56.1`** | 최신 |

`transformers` 한 줄만 봐도 sa3(≥5.8)와 flux(==4.56.1)는 한 환경에 공존할 수 없다.
한 커널에서 노트북 4개를 돌리면 pip이 서로의 torch/transformers를 덮어써서 전부 깨진다.

## 왜 pip이 아니라 uv인가

`ace`의 `torch==2.10.0+cu128` 과 `nano-vllm` 은 `[tool.uv.index]` / `[tool.uv.sources]` 에만
선언돼 있다. **pip은 그 설정을 읽지 않으므로** `pip install git+...` 는 의존성 해결에 실패한다.
`sa3` 도 마찬가지로 cu126 인덱스가 `[tool.uv.sources]` 로 매핑돼 있다.

## 실행 방법

위에서부터 순서대로 실행. **이 노트북만 아무 커널로나 돌리면 된다** (기본 python3).
한 환경이 실패해도 나머지는 계속 진행되고, 마지막 셀이 요약을 출력한다.

## 0. 설정 + 사전 점검

`PROJ_BASE` 만 환경에 맞게 고치면 된다.

In [ ]:
import os, sys, json, shutil, subprocess, pathlib, getpass, socket

USER = getpass.getuser()

# ---- 이 노드에서 유효한 루트를 '직접 찾는다' ------------------------------
# 학과 클러스터는 같은 저장소가 노드마다 다른 경로에 마운트된다.
#   로그인 노드(hpcmaster) : /data1/$USER   (= HOME)
#   계산 노드(hpc-stat1)   : /mnt/data1/$USER
# /root 는 쓸 수 없다. 예전 기본값 "/root/mystery" 가 PermissionError 의 원인이었다.
def pick_root():
    for c in (f"/mnt/data1/{USER}", f"/data1/{USER}",
              str(pathlib.Path.home()), "/workspace", "/scratch"):
        if pathlib.Path(c).is_dir() and os.access(c, os.W_OK):
            return c
    return None

ROOT = os.environ.get("PROJ_ROOT") or pick_root()
if ROOT is None:
    raise SystemExit(
        "쓸 수 있는 루트를 못 찾았다.\n"
        f"  확인한 곳: /mnt/data1/{USER}, /data1/{USER}, {pathlib.Path.home()}, /workspace, /scratch\n"
        "  이 셀 위에서 os.environ['PROJ_ROOT'] = '/쓰기가능한/경로' 로 지정할 것."
    )

# 계산 노드에는 $HOME 이 아예 없다. 두면 pip cache / kernelspec 등록이 전부 실패한다.
if not pathlib.Path(os.environ.get("HOME", "/nonexistent")).is_dir():
    os.environ["HOME"] = ROOT
    print("HOME 이 없어서 재설정:", ROOT)

PROJ_BASE  = os.environ.get("PROJ_BASE")  or f"{ROOT}/mystery"
PROJ_INPUT = os.environ.get("PROJ_INPUT") or os.getcwd()
HF_HOME    = os.environ.get("HF_HOME")    or f"{PROJ_BASE}/hf_cache"

REPOS   = f"{PROJ_BASE}/repos"
ENVS    = f"{PROJ_BASE}/envs"
LOGS    = f"{PROJ_BASE}/logs"
RESULTS = {}

# ---- 캐시/임시파일을 전부 PROJ_BASE 로 -----------------------------------
# uv 는 기본적으로 ~/.cache/uv 에 받아 압축을 푼다. 그 파일시스템이 작으면
#   "failed to write ... libtorch_cuda.so: No space left on device (os error 28)"
# 로 죽는다. 캐시와 venv 가 같은 파일시스템이어야 하드링크가 되어 공간도 아낀다.
UV_CACHE = f"{PROJ_BASE}/uv_cache"
TMPDIR   = f"{PROJ_BASE}/tmp"
os.environ["UV_CACHE_DIR"]  = UV_CACHE
os.environ["TMPDIR"]        = TMPDIR
os.environ["PIP_CACHE_DIR"] = f"{PROJ_BASE}/pip_cache"
os.environ["MPLCONFIGDIR"]  = f"{PROJ_BASE}/mpl_cache"
os.environ["HF_HOME"]       = HF_HOME


def run(cmd, cwd=None, log=None, quiet=False):
    """명령을 돌리고 (성공여부, 마지막 몇 줄). 실패해도 예외를 안 던진다."""
    shell = isinstance(cmd, str)
    p = subprocess.run(["bash", "-lc", cmd] if shell else cmd,
                       cwd=cwd, capture_output=True, text=True)
    out = (p.stdout or "") + (p.stderr or "")
    if log:
        pathlib.Path(log).parent.mkdir(parents=True, exist_ok=True)
        with open(log, "a") as f:
            f.write(f"\n$ {cmd if shell else ' '.join(map(str, cmd))}\n{out}")
    tail = out.strip().splitlines()[-12:]
    if not quiet:
        for line in tail:
            print("   |", line[:160])
    return p.returncode == 0, tail


def free_gb(path):
    try:
        return shutil.disk_usage(path).free / 1e9
    except OSError:
        return -1.0


def need_space(gb, where=None):
    """공간이 모자라면 가중치를 받기 전에 멈춘다."""
    where = where or PROJ_BASE
    have = free_gb(where)
    if have < gb:
        raise SystemExit(
            f"디스크 부족: {where} 여유 {have:.0f}GB < 필요 {gb}GB\n"
            f"  1) PROJ_BASE 를 더 큰 볼륨으로 바꾸고 이 노트북을 처음부터 다시 실행\n"
            f"  2) 또는 아래 1-A 정리 셀 실행"
        )
    return have


print("host   :", socket.gethostname(), "| user:", USER)
print("python :", sys.version.split()[0], "|", sys.executable)
print("cwd    :", os.getcwd())
print()

print("=== 마운트별 여유 공간 ===")
run("df -h --output=target,size,avail,pcent 2>/dev/null | sort -k3 -hr | head -12")
print()

# ---- 쓰기 권한: 실패하면 '여기서 멈춘다' --------------------------------
# 예전에는 실패를 출력만 하고 넘어가서, 뒤 셀이 /root/mystery 를 계속 건드리다
# PermissionError 로 죽었다.
try:
    for d in (PROJ_BASE, REPOS, ENVS, LOGS, HF_HOME, UV_CACHE, TMPDIR):
        pathlib.Path(d).mkdir(parents=True, exist_ok=True)
except (PermissionError, OSError) as e:
    raise SystemExit(
        f"{PROJ_BASE} 에 쓸 수 없다 -> {type(e).__name__}: {e}\n"
        f"  이 셀 위에서 지정할 것:\n"
        f"      os.environ['PROJ_BASE'] = '/data1/{USER}/mystery'\n"
        f"  (위 df 표에서 여유가 큰 마운트를 고를 것)"
    )

print(f"ROOT      : {ROOT}")
print(f"PROJ_BASE : {PROJ_BASE}   (쓰기 OK)")
print(f"PROJ_INPUT: {PROJ_INPUT}")
print(f"HF_HOME   : {HF_HOME}")
print(f"UV_CACHE  : {UV_CACHE}   <- 작은 디스크면 os error 28")
print()

for f in ("asset_input_cinderella_en.json", "asset_input_heungbu_en.json"):
    q = pathlib.Path(PROJ_INPUT) / f
    print(f"입력 JSON : {'OK  ' if q.exists() else '없음'} {q}")

ok, tail = run("nvidia-smi --query-gpu=name,memory.total,compute_cap,driver_version "
               "--format=csv,noheader", quiet=True)
print()
if ok and tail:
    for line in tail:
        print("GPU       :", line)
    try:
        cc = float(tail[0].split(",")[2])
        print(f"            FP8(CC>=8.9): {'가능' if cc >= 8.9 else '불가 — FLUX.2 klein 못 돌림'}")
    except (ValueError, IndexError):
        pass
else:
    print("GPU       : 없음 (로그인 노드면 정상. 설치에는 GPU 가 필요 없다)")

here = free_gb(PROJ_BASE)
print()
print(f"PROJ_BASE 여유 : {here:.0f} GB")
print( "  필요량 대략   : 환경 4개 약 50GB + 가중치 약 120GB = 170GB 이상 권장")
if here < 170:
    print("  ** 부족하다. PROJ_BASE 를 더 큰 볼륨으로 바꾸고 이 셀부터 다시 실행할 것. **")

## 1. uv 설치

`ace` / `sa3` 의 공식 설치 방식이 `uv sync` 다. pip 으로는 설치가 안 된다(위 설명 참고).

## 1-A. 디스크 정리 (No space left on device 를 만났다면)

작은 디스크에 이미 쌓인 캐시를 비운다. 실패한 `.venv` 도 지우고 다시 만든다.


In [ ]:
# 기본 캐시 위치(작은 디스크)에 쌓인 것들을 비운다.
print("=== 정리 전 ===")
run("du -sh ~/.cache/uv ~/.cache/pip ~/.cache/huggingface 2>/dev/null; "
    "df -h "+PROJ_BASE+" 2>/dev/null | tail -2")

run("uv cache clean 2>/dev/null; rm -rf ~/.cache/uv ~/.cache/pip", quiet=True)

# 앞선 시도에서 만들다 만 venv 는 지운다. 반쯤 깔린 상태로 재개하면 더 꼬인다.
# 권한 없는 경로(예전 기본값 /root/mystery)는 건너뛴다.
for d in (f"{REPOS}/stable-audio-3/.venv", f"{REPOS}/ACE-Step-1.5/.venv",
          f"{REPOS}/flux2/.venv", f"{ENVS}/img"):
    try:
        if pathlib.Path(d).exists():
            print("삭제:", d)
            shutil.rmtree(d, ignore_errors=True)
    except (PermissionError, OSError) as e:
        print("건너뜀:", d, "->", type(e).__name__, e)

print("\n=== 정리 후 ===")
print(f"PROJ_BASE 여유: {free_gb(PROJ_BASE):.0f} GB   ({PROJ_BASE})")

In [ ]:
ok, _ = run("command -v uv >/dev/null && uv --version", quiet=True)
if not ok:
    print("uv 없음 → 설치")
    run("curl -LsSf https://astral.sh/uv/install.sh | sh", log=f"{LOGS}/uv.log")

# 이 커널 프로세스의 PATH 에 uv 위치를 넣어준다 (설치 직후엔 안 잡혀 있다)
for extra in (f"{pathlib.Path.home()}/.local/bin", f"{pathlib.Path.home()}/.cargo/bin"):
    if extra not in os.environ["PATH"]:
        os.environ["PATH"] = extra + ":" + os.environ["PATH"]

ok, tail = run("uv --version", quiet=True)
print("uv :", tail[0] if ok and tail else "FAIL — 아래 로그 확인")
if not ok:
    print(open(f"{LOGS}/uv.log").read()[-2000:])

## 2. 커널 등록 헬퍼

`kernel.json` 에 `PROJ_BASE` / `PROJ_INPUT` / `HF_HOME` 을 직접 박아 넣는다.
이렇게 해두면 노트북의 `os.environ.get("PROJ_BASE", "/root/mystery")` 가 항상 잡혀서
경로 문제가 다시 안 생긴다.

HF 토큰은 여기 넣지 않는다. 노트북에도 적지 말 것 — 터미널에서 `export HF_TOKEN=...`.

In [ ]:
KERNEL_DIR = pathlib.Path.home() / ".local/share/jupyter/kernels"

def register_kernel(py, name, display):
    """venv 의 python 으로 커널을 등록하고 kernel.json 에 환경변수를 주입한다."""
    if not pathlib.Path(py).exists():
        return False, [f"python 없음: {py}"]
    ok, tail = run([py, "-m", "ipykernel", "install", "--user",
                    "--name", name, "--display-name", display],
                   log=f"{LOGS}/{name}.log", quiet=True)
    if not ok:
        return False, tail
    kj = KERNEL_DIR / name / "kernel.json"
    try:
        spec = json.loads(kj.read_text())
        spec.setdefault("env", {}).update({
            "PROJ_BASE":  PROJ_BASE,
            "PROJ_INPUT": PROJ_INPUT,
            "HF_HOME":    HF_HOME,
        })
        kj.write_text(json.dumps(spec, indent=1))
        return True, [f"kernel.json patched: {kj}"]
    except Exception as ex:
        return False, [f"{type(ex).__name__}: {ex}"]


def finish(name, ok, tail, where=""):
    RESULTS[name] = ("OK" if ok else "FAIL", where if ok else f"로그: {LOGS}/{name}.log")
    print(("  OK   " if ok else "  FAIL ") + name + ("  " + where if ok else ""))
    if not ok:
        for line in tail:
            print("       |", line[:160])

print("헬퍼 준비됨. 커널 등록 위치:", KERNEL_DIR)

## 3. sa3 — Stable Audio 3

노트북이 하던 `pip install torch==2.7.1 --index-url .../whl/cu124` 는 **불가능**하다.
cu124 인덱스의 최신 torch 는 2.6.0 이고 2.7.1 이 아예 없다
(`No matching distribution found`). 공식 pyproject 는 cu126 인덱스를 쓴다.

In [ ]:
name, D, LOG = "sa3", f"{REPOS}/stable-audio-3", f"{LOGS}/sa3.log"
need_space(25)    # 공간 부족이면 받기 전에 멈춘다
print("## sa3 — Stable Audio 3")

ok, tail = run(f"[ -d {D}/.git ] || git clone --depth 1 "
               f"https://github.com/Stability-AI/stable-audio-3.git {D}", log=LOG, quiet=True)
if ok:
    print("  uv sync ... (몇 분 걸린다)")
    ok, tail = run("uv sync", cwd=D, log=LOG, quiet=True)
if ok:
    ok, tail = run("uv pip install ipykernel", cwd=D, log=LOG, quiet=True)
if ok:
    ok, tail = register_kernel(f"{D}/.venv/bin/python", name, "sa3 · Stable Audio 3")

finish(name, ok, tail, f"{D}/.venv")

## 4. ace — ACE-Step 1.5

`pip install git+...` 가 실패하는 이유: pyproject 가 linux x86_64 에서
`torch==2.10.0+cu128` 을 요구하는데 그 버전은 PyPI 가 아니라
`[tool.uv.index]` 로 선언된 cu128 인덱스에만 있다. `nano-vllm` 도 `[tool.uv.sources]` 매핑이다.

In [ ]:
name, D, LOG = "ace", f"{REPOS}/ACE-Step-1.5", f"{LOGS}/ace.log"
need_space(30)    # 공간 부족이면 받기 전에 멈춘다
print("## ace — ACE-Step 1.5")

ok, tail = run(f"[ -d {D}/.git ] || git clone --depth 1 "
               f"https://github.com/ace-step/ACE-Step-1.5.git {D}", log=LOG, quiet=True)
if ok:
    print("  uv sync ... (제일 오래 걸린다. 10분 이상 가능)")
    ok, tail = run("uv sync", cwd=D, log=LOG, quiet=True)
if ok:
    ok, tail = run("uv pip install ipykernel", cwd=D, log=LOG, quiet=True)
if ok:
    ok, tail = register_kernel(f"{D}/.venv/bin/python", name, "ace · ACE-Step 1.5")

finish(name, ok, tail, f"{D}/.venv")

## 5. flux — FLUX.2 klein

공식: python 3.12 venv + `pip install -e . --extra-index-url .../cu129`.
pyproject 가 `torch==2.8.0`, `transformers==4.56.1` 을 못박아 둔다.

**GPU compute capability 가 8.9 미만이면 이 커널을 만들어도 klein 은 못 돌린다**
(텍스트 인코더가 Qwen3-4B-FP8). 0번 셀 출력의 CC 값을 확인할 것.

In [ ]:
name, D, LOG = "flux", f"{REPOS}/flux2", f"{LOGS}/flux.log"
need_space(25)    # 공간 부족이면 받기 전에 멈춘다
print("## flux — FLUX.2 klein")

ok, tail = run(f"[ -d {D}/.git ] || git clone --depth 1 "
               f"https://github.com/black-forest-labs/flux2.git {D}", log=LOG, quiet=True)
if ok:
    ok, tail = run("uv venv --python 3.12 .venv", cwd=D, log=LOG, quiet=True)
if ok:
    print("  uv pip install -e . (cu129) ...")
    ok, tail = run("uv pip install --python .venv/bin/python -e . "
                   "--extra-index-url https://download.pytorch.org/whl/cu129",
                   cwd=D, log=LOG, quiet=True)
if ok:
    ok, tail = run("uv pip install --python .venv/bin/python ipykernel pillow",
                   cwd=D, log=LOG, quiet=True)
if ok:
    ok, tail = register_kernel(f"{D}/.venv/bin/python", name, "flux · FLUX.2 klein")

finish(name, ok, tail, f"{D}/.venv")

## 6. img — HiDream-I1-Dev (diffusers)

이 환경만 공식 lock 파일이 없어 버전을 직접 고른다. torch 는 CUDA 인덱스에서,
나머지는 PyPI 에서 받도록 호출을 나눴다 (인덱스를 섞으면 CPU 빌드가 깔릴 수 있다).

노트북이 쓰던 `HiDreamO1ImagePipeline` / `HiDream-O1-Image-Dev` 는 diffusers 에
**존재하지 않는 이름**이다. 문서상 클래스는 `HiDreamImagePipeline` 하나뿐이고
지원 체크포인트는 `HiDream-I1-Full/Dev/Fast` 다.

In [ ]:
name, D, LOG = "img", f"{ENVS}/img", f"{LOGS}/img.log"
need_space(25)    # 공간 부족이면 받기 전에 멈춘다
print("## img — HiDream-I1-Dev (diffusers)")

ok, tail = run(f"uv venv --python 3.12 {D}", log=LOG, quiet=True)
if ok:
    print("  torch (cu126) ...")
    ok, tail = run(f"uv pip install --python {D}/bin/python torch==2.8.0 torchvision==0.23.0 "
                   f"--index-url https://download.pytorch.org/whl/cu126", log=LOG, quiet=True)
if ok:
    print("  diffusers 계열 ...")
    ok, tail = run(f"uv pip install --python {D}/bin/python "
                   f"'diffusers>=0.39.0' transformers accelerate safetensors "
                   f"sentencepiece protobuf pillow ipykernel", log=LOG, quiet=True)
if ok:
    ok, tail = register_kernel(f"{D}/bin/python", name, "img · HiDream-I1 (diffusers)")

finish(name, ok, tail, D)

## 7. 검증

설치됐다고 믿지 말고 **실제로 import 해본다.** 여기서 OK 가 떠야 노트북을 돌릴 수 있다.

In [ ]:
CHECKS = {
    "sa3":  (f"{REPOS}/stable-audio-3/.venv/bin/python", """
import torch, transformers
from stable_audio_3 import StableAudioModel
print(f"torch={torch.__version__} cuda={torch.cuda.is_available()} "
      f"transformers={transformers.__version__} StableAudioModel=OK")
"""),
    "ace":  (f"{REPOS}/ACE-Step-1.5/.venv/bin/python", """
import torch, transformers
from acestep.inference import generate_music, GenerationParams, GenerationConfig
print(f"torch={torch.__version__} cuda={torch.cuda.is_available()} "
      f"transformers={transformers.__version__} generate_music=OK")
"""),
    "flux": (f"{REPOS}/flux2/.venv/bin/python", """
import torch, transformers
print(f"torch={torch.__version__} cuda={torch.cuda.is_available()} "
      f"transformers={transformers.__version__}")
"""),
    "img":  (f"{ENVS}/img/bin/python", """
import torch, diffusers, transformers
from diffusers import HiDreamImagePipeline
print(f"torch={torch.__version__} cuda={torch.cuda.is_available()} "
      f"diffusers={diffusers.__version__} transformers={transformers.__version__} "
      f"HiDreamImagePipeline=OK")
"""),
}

print(f"{'커널':6s} 결과")
print("-" * 78)
for name, (py, snippet) in CHECKS.items():
    if not pathlib.Path(py).exists():
        print(f"{name:6s} (환경 없음 — 해당 셀을 먼저 돌릴 것)")
        continue
    p = subprocess.run([py, "-c", snippet], capture_output=True, text=True)
    out = ((p.stdout or "") + (p.stderr or "")).strip().splitlines()
    print(f"{name:6s} {out[-1][:100] if out else '(출력 없음)'}")
    if p.returncode != 0:
        for line in out[-6:-1]:
            print(f"       | {line[:100]}")

## 8. 요약 + 다음 할 일

In [ ]:
print("환경 설치 결과")
print("-" * 78)
for k in ("sa3", "ace", "flux", "img"):
    st, where = RESULTS.get(k, ("미실행", ""))
    print(f"  {k:6s} {st:6s} {where}")

print()
print("등록된 커널")
print("-" * 78)
if KERNEL_DIR.exists():
    for d in sorted(KERNEL_DIR.iterdir()):
        print("  ", d.name)
else:
    print("   (없음)")

print(f"""
다음 할 일
  1. Jupyter 를 새로고침해야 새 커널이 목록에 뜬다.
  2. 각 노트북에서 커널을 위 이름으로 바꾼다 (ace / sa3 / flux / img).
  3. 노트북의 1번 "설치" 셀은 이제 실행하지 말 것.
     그 셀이 pip 으로 torch 를 다시 깔면 방금 맞춘 버전이 도로 깨진다.
  4. HF 토큰은 노트북에 적지 말고 터미널에서:
       export HF_TOKEN=hf_xxx
     사전 승인이 필요한 gated repo:
       - stabilityai/stable-audio-3-medium   (약관 동의)
       - meta-llama/Llama-3.1-8B-Instruct    (수동 승인, 시간이 걸린다)
         승인이 늦으면 NousResearch/Meta-Llama-3.1-8B-Instruct 미러 사용 가능

커널에 박힌 환경변수 (노트북이 자동으로 따라간다)
  PROJ_BASE  = {PROJ_BASE}
  PROJ_INPUT = {PROJ_INPUT}
  HF_HOME    = {HF_HOME}
""")